# 00setup: project setup and data connection tests

PF5: haze from the 2026 Indonesian fires and Super Typhoon Choi-wan.

This notebook does three things:

1. Creates the project folders.
2. Reads the private keys from `config/keys.toml` (never printed).
3. Tests each data source with a tiny request for the test day:
   NASA Earthdata (MERRA-2), GEOS-FP (the model behind earth.nullschool.net), NASA FIRMS fire hotspots, ERA5 on the Copernicus CDS, and CAMS on the Copernicus ADS.

Each test reports OK or FAIL and the notebook carries on, so one broken connection does not stop the others. The summary at the end lists all results.

## Section 0: settings

In [1]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
KEYS_FILE = PROJECT_DIR / "config" / "keys.toml"
TEST_DIR = PROJECT_DIR / "data" / "test"

TEST_DATE = "2026-10-03"  # <- day of the haze streamer into Choi-wan

# Study box in degrees (east positive, north positive)
BOX_WEST = 90.0  # <-
BOX_EAST = 160.0  # <-
BOX_SOUTH = -15.0  # <-
BOX_NORTH = 45.0  # <-

## Section 1: imports and folders

In [2]:
import io
import tomllib
import zipfile
import requests
import numpy as np
import pandas as pd
import xarray as xr
import earthaccess
import cdsapi

FOLDERS = ["config", "data/raw", "data/processed", "data/test", "docs"]
for sub in FOLDERS:
    (PROJECT_DIR / sub).mkdir(parents=True, exist_ok=True)

RESULTS = {}
YEAR, MONTH, DAY = TEST_DATE.split("-")
print("Project folder:", PROJECT_DIR)
print("Test day:", TEST_DATE)

Project folder: .
Test day: 2026-10-03


.\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Section 2: read the private keys

The keys stay in `config/keys.toml`, which is excluded from GitHub by `.gitignore`. This cell only reports whether each key is filled in.

In [3]:
with open(KEYS_FILE, "rb") as f:
    KEYS = tomllib.load(f)

for section in ["firms", "cds", "ads"]:
    filled = len(KEYS.get(section, {}).get("key", "").strip()) > 0
    print(f"{section:6s} key filled in: {filled}")

firms  key filled in: True
cds    key filled in: True
ads    key filled in: True


## Section 3: NASA Earthdata and MERRA-2

The first time, `earthaccess` asks for your Earthdata username and password (a box at the top of VS Code) and saves them in `_netrc` in your Windows user folder. After that it logs in silently.

The test searches for the MERRA-2 hourly aerosol file of the test day, then opens the first bytes of it. That second step only works if the NASA GESDISC DATA ARCHIVE application is approved in your Earthdata profile.

In [4]:
try:
    auth = earthaccess.login(persist=True)
    print("Earthdata login:", auth.authenticated)
    recent = earthaccess.search_data(short_name="M2T1NXAER", temporal=("2026-08-01", "2026-12-31"))
    days = sorted(g["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"][:10] for g in recent)
    print("MERRA-2 aerosol days found since 1 Aug 2026:", len(days))
    print("Latest MERRA-2 aerosol day:", days[-1] if days else "none")
    test_granules = [g for g in recent if g["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"][:10] == TEST_DATE]
    print("Test day available in MERRA-2:", len(test_granules) > 0)
    if test_granules:
        url = test_granules[0].data_links()[0]
        session = earthaccess.get_requests_https_session()
        r = session.get(url, stream=True, timeout=60)
        print("Download access to GES DISC, HTTP status:", r.status_code)
        r.close()
        RESULTS["Earthdata / MERRA-2"] = "OK" if r.status_code == 200 else f"FAIL (HTTP {r.status_code})"
    else:
        RESULTS["Earthdata / MERRA-2"] = f"LOGIN OK, test day not released yet (latest {days[-1] if days else 'none'})"
except Exception as e:
    print("Earthdata test failed:", type(e).__name__, e)
    RESULTS["Earthdata / MERRA-2"] = f"FAIL ({type(e).__name__})"

Earthdata login: True
MERRA-2 aerosol days found since 1 Aug 2026: 62
Latest MERRA-2 aerosol day: 2026-10-01
Test day available in MERRA-2: False


## Section 4: GEOS-FP through OPeNDAP

GEOS-FP is the NASA model shown on earth.nullschool.net. It is served openly (no login) by NASA's NCCS OPeNDAP server, which lets us cut out just our box instead of downloading global files. The test reads total aerosol optical depth (`totexttau`) for one time step.

Two reading libraries are tried in turn: `netCDF4` (fast, but its web reader often fails on Windows) and `pydap` (pure Python, uses the same web connection as the FIRMS test).

In [5]:
GEOSFP_URL = "https://opendap.nccs.nasa.gov/dods/GEOS-5/fp/0.25_deg/assim/tavg3_2d_aer_Nx"
GEOSFP_ENGINES = ["netcdf4", "pydap"]  # <- reading libraries, tried in this order
geos = None
try:
    info = requests.get(GEOSFP_URL + ".info", timeout=60)
    print("GEOS-FP server reachable, HTTP status:", info.status_code)
except Exception as e:
    print("GEOS-FP server not reachable:", type(e).__name__)
for engine in GEOSFP_ENGINES:
    try:
        url = GEOSFP_URL.replace("https://", "dap2://") if engine == "pydap" else GEOSFP_URL
        geos = xr.open_dataset(url, engine=engine)
        GEOSFP_ENGINE = engine
        print("Opened with engine:", engine)
        break
    except Exception as e:
        print(f"Engine {engine} failed:", type(e).__name__, str(e)[:200])
try:
    if geos is None:
        raise OSError("no engine could open GEOS-FP")
    geos_times = pd.to_datetime([str(t) for t in geos.time.values])
    print("GEOS-FP time range:", geos_times[0], "to", geos_times[-1])
    i = int(np.abs(geos_times - pd.Timestamp(f"{TEST_DATE} 04:30")).argmin())
    print("Time step used:", geos_times[i])
    aod = geos["totexttau"].isel(time=i).sel(lat=slice(BOX_SOUTH, BOX_NORTH), lon=slice(BOX_WEST, BOX_EAST)).load()
    print("Grid cells in box:", aod.shape)
    print("Highest aerosol optical depth in box:", round(float(aod.max()), 2))
    aod.to_netcdf(TEST_DIR / "geosfp_test.nc")
    RESULTS["GEOS-FP (OPeNDAP)"] = f"OK ({GEOSFP_ENGINE})"
except Exception as e:
    print("GEOS-FP test failed:", type(e).__name__, e)
    RESULTS["GEOS-FP (OPeNDAP)"] = f"FAIL ({type(e).__name__})"


GEOS-FP server reachable, HTTP status: 200
Engine netcdf4 failed: OSError [Errno -68] NetCDF: I/O failure: 'https://opendap.nccs.nasa.gov/dods/GEOS-5/fp/0.25_deg/assim/tavg3_2d_aer_Nx'


.\.venv\Lib\site-packages\xarray\coding\times.py:215: SerializationWarning: Ambiguous reference date string: 1-1-1 00:00:0.0. The first value is assumed to be the year hence will be padded with zeros to remove the ambiguity (the padded reference date string is: 0001-1-1 00:00:0.0). To remove this message, remove the ambiguity by padding your reference date strings with zeros.
  ref_date = _ensure_padded_year(ref_date)


Opened with engine: pydap
GEOS-FP time range: 2017-12-01 01:30:00 to 2026-10-29 07:30:00
Time step used: 2026-10-03 04:30:00
Grid cells in box: (241, 225)
Highest aerosol optical depth in box: 4.14


.\.venv\Lib\site-packages\xarray\coding\times.py:215: SerializationWarning: Ambiguous reference date string: 1-1-1 00:00:0.0. The first value is assumed to be the year hence will be padded with zeros to remove the ambiguity (the padded reference date string is: 0001-1-1 00:00:0.0). To remove this message, remove the ambiguity by padding your reference date strings with zeros.
  ref_date = _ensure_padded_year(ref_date)


## Section 5: NASA FIRMS fire hotspots

First the key status (how many requests are left in the current 10 minute window), then one day of VIIRS hotspots for the box. Error messages print only the HTTP status, never the address, because the address contains the key.

In [6]:
FIRMS_KEY = KEYS["firms"]["key"]
FIRMS_SOURCE = "VIIRS_SNPP_NRT"  # <- also possible: VIIRS_NOAA20_NRT, VIIRS_NOAA21_NRT, MODIS_NRT
try:
    status_url = "https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=" + FIRMS_KEY
    s = requests.get(status_url, timeout=60)
    print("Key status, HTTP status:", s.status_code)
    if s.status_code == 200:
        info = s.json()
        print("Requests used / limit per 10 minutes:", info.get("current_transactions"), "/", info.get("transaction_limit"))
    box_text = f"{BOX_WEST:g},{BOX_SOUTH:g},{BOX_EAST:g},{BOX_NORTH:g}"
    area_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{FIRMS_KEY}/{FIRMS_SOURCE}/{box_text}/1/{TEST_DATE}"
    r = requests.get(area_url, timeout=120)
    print("Hotspot request, HTTP status:", r.status_code)
    fires = pd.read_csv(io.StringIO(r.text))
    if "latitude" not in fires.columns:
        raise ValueError("FIRMS returned a message instead of data: " + r.text[:200].replace(FIRMS_KEY, "***"))
    print("Hotspots in box on", TEST_DATE, ":", len(fires))
    print(fires[["latitude", "longitude", "acq_date", "acq_time", "frp", "confidence"]].head())
    fires.to_csv(TEST_DIR / "firms_test.csv", index=False)
    RESULTS["FIRMS"] = "OK"
except Exception as e:
    print("FIRMS test failed:", type(e).__name__, str(e).replace(FIRMS_KEY, "***"))
    RESULTS["FIRMS"] = f"FAIL ({type(e).__name__})"

Key status, HTTP status: 200
Requests used / limit per 10 minutes: 0 / 5000
Hotspot request, HTTP status: 200
Hotspots in box on 2026-10-03 : 8313
   latitude  longitude    acq_date  acq_time    frp confidence
0  -4.41621  152.11021  2026-10-03       241   3.79          n
1  -4.31815  152.10938  2026-10-03       241   3.83          n
2  -4.23477  152.04686  2026-10-03       241  13.37          n
3  -3.99474  152.85268  2026-10-03       241   4.65          n
4  -3.95538  152.80782  2026-10-03       241   6.23          n


## Section 6: ERA5 on the Copernicus Climate Data Store

One hour of 10 m wind for the box. Small requests usually finish in one to five minutes, depending on the CDS queue.

In [7]:
era5_file = TEST_DIR / "era5_test.nc"
era5_request = {}
era5_request["product_type"] = ["reanalysis"]
era5_request["variable"] = ["10m_u_component_of_wind", "10m_v_component_of_wind"]
era5_request["year"] = [YEAR]
era5_request["month"] = [MONTH]
era5_request["day"] = [DAY]
era5_request["time"] = ["00:00"]
era5_request["data_format"] = "netcdf"
era5_request["download_format"] = "unarchived"
era5_request["area"] = [BOX_NORTH, BOX_WEST, BOX_SOUTH, BOX_EAST]
try:
    cds = cdsapi.Client(url=KEYS["cds"]["url"], key=KEYS["cds"]["key"], quiet=True, progress=False, retry_max=5, sleep_max=10)
    cds.retrieve("reanalysis-era5-single-levels", era5_request, str(era5_file))
    era5 = xr.open_dataset(era5_file)
    speed = np.hypot(era5["u10"], era5["v10"])
    print("ERA5 grid:", dict(era5.sizes))
    print("Highest 10 m wind speed in box (m/s):", round(float(speed.max()), 1))
    era5.close()
    RESULTS["ERA5 (CDS)"] = "OK"
except Exception as e:
    print("ERA5 test failed:", type(e).__name__, e)
    RESULTS["ERA5 (CDS)"] = f"FAIL ({type(e).__name__})"

ERA5 grid: {'valid_time': 1, 'latitude': 241, 'longitude': 281}
Highest 10 m wind speed in box (m/s): 24.0


## Section 7: CAMS on the Copernicus Atmosphere Data Store

One time step of the CAMS PM2.5 analysis (forecast lead time 0) for the box. CAMS stores PM2.5 in kg per cubic metre; times 1e9 gives micrograms per cubic metre.

In [8]:
cams_zip = TEST_DIR / "cams_test.zip"
cams_request = {}
cams_request["variable"] = ["particulate_matter_2.5um"]
cams_request["date"] = [f"{TEST_DATE}/{TEST_DATE}"]
cams_request["time"] = ["00:00"]
cams_request["leadtime_hour"] = ["0"]
cams_request["type"] = ["forecast"]
cams_request["data_format"] = "netcdf_zip"
cams_request["area"] = [BOX_NORTH, BOX_WEST, BOX_SOUTH, BOX_EAST]
try:
    ads = cdsapi.Client(url=KEYS["ads"]["url"], key=KEYS["ads"]["key"], quiet=True, progress=False, retry_max=5, sleep_max=10)
    ads.retrieve("cams-global-atmospheric-composition-forecasts", cams_request, str(cams_zip))
    if zipfile.is_zipfile(cams_zip):
        with zipfile.ZipFile(cams_zip) as z:
            nc_name = [n for n in z.namelist() if n.endswith(".nc")][0]
            z.extract(nc_name, TEST_DIR)
        cams_file = TEST_DIR / nc_name
    else:
        cams_file = cams_zip
    cams = xr.open_dataset(cams_file)
    pm25 = cams["pm2p5"] * 1e9
    print("CAMS grid:", dict(cams.sizes))
    print("Highest PM2.5 in box (micrograms per m3):", round(float(pm25.max()), 0))
    cams.close()
    RESULTS["CAMS (ADS)"] = "OK"
except Exception as e:
    print("CAMS test failed:", type(e).__name__, e)
    RESULTS["CAMS (ADS)"] = f"FAIL ({type(e).__name__})"

CAMS grid: {'forecast_period': 1, 'forecast_reference_time': 1, 'latitude': 150, 'longitude': 176}
Highest PM2.5 in box (micrograms per m3): 434.0


## Section 8: summary

In [9]:
summary = pd.DataFrame(list(RESULTS.items()), columns=["Data source", "Result"])
print(summary.to_string(index=False))

        Data source                                                  Result
Earthdata / MERRA-2 LOGIN OK, test day not released yet (latest 2026-10-01)
  GEOS-FP (OPeNDAP)                                          FAIL (OSError)
              FIRMS                                                      OK
         ERA5 (CDS)                                                      OK
         CAMS (ADS)                                                      OK


___
**END**